Import ชุดคำสั่งที่จำเป็น

In [ ]:
import pandas as pd
import numpy as np

# for reading and displaying images
from skimage.io import imread
import matplotlib.pyplot as plt

# for creating validation set
from sklearn.model_selection import train_test_split

# for evaluating the model
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# PyTorch libraries and modules
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import Adam, SGD


Data Loader

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/Deep Learning/ThaiCharacter Dataset.zip" -d /content/dataset

# จากนั้นตั้งตัวแปร path สำหรับใช้ต่อในโค้ด:

DATA_PATH = "/content/dataset/round2"

In [ ]:
import os

DATA_PATH = "/content/dataset/round2"
classes = sorted([d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))])

Data Loader + Data Augmentation (วนอ่านภาพจริงพร้อมเติมภาพให้คลาสที่ขาด)

In [ ]:
from skimage.transform import resize, rotate
from skimage.util import random_noise

def augment_image(img):
    angle = np.random.uniform(-15, 15)  # เทียบเท่า RandomRotation
    img_aug = rotate(img, angle, mode='edge')
    if np.random.rand() < 0.5:
        img_aug = random_noise(img_aug, var=0.005)  # เพิ่ม noise เบาๆ
    return img_aug

In [ ]:
MIN_SAMPLES_PER_CLASS = 50
IMG_SIZE = 224  # ทดลอง: ขนาดเท่ากับที่ ResNet18 pretrained มา (จากเดิม 64)

# เปลี่ยนวิธีโหลด: เดิมโหลด+resize ภาพทั้งหมดเข้า RAM ทีเดียว (63,000+ ภาพ x 224x224 x float32
# ~12GB+ ต่อก้อน) ทำให้ Colab ฟรี RAM ไม่พอ crash ตอน IMG_SIZE=224 -> เก็บแค่ "path + label" ไว้ก่อน
# (เบามาก) แล้วให้ Dataset โหลด+ประมวลผลทีละภาพตอนเทรนจริงแทน (lazy loading, ดู cell ถัดไป)

samples = []  # (path, augment_flag, label)

for cls in tqdm(classes):
    cls_folder = os.path.join(DATA_PATH, cls)
    img_files = [f for f in os.listdir(cls_folder) if f.lower().endswith('.jpg')]
    img_paths = [os.path.join(cls_folder, f) for f in img_files]

    for p in img_paths:
        samples.append((p, False, cls))

    # เติม path ซ้ำให้คลาสที่ขาด (จะ augment ตอนโหลดจริงใน Dataset ไม่ใช่ตอนนี้)
    n_needed = MIN_SAMPLES_PER_CLASS - len(img_paths)
    if n_needed > 0:
        for i in range(n_needed):
            base_path = img_paths[i % len(img_paths)]
            samples.append((base_path, True, cls))

print(f"รวม {len(samples)} samples ({len(classes)} คลาส)")

Training & Validating Set Generation

In [ ]:
# แปลง label เป็น class index
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
sample_paths = [s[0] for s in samples]
sample_aug = [s[1] for s in samples]
sample_labels = np.array([class_to_idx[s[2]] for s in samples])

# stratify กัน val set สุ่มไม่สมดุลระหว่าง 72 คลาส (มีบางคลาสตัวอย่างน้อย)
idx_all = np.arange(len(samples))
train_idx, val_idx = train_test_split(
    idx_all, test_size=0.2, stratify=sample_labels, random_state=42
)

train_samples = [(sample_paths[i], sample_aug[i], int(sample_labels[i])) for i in train_idx]
val_samples = [(sample_paths[i], sample_aug[i], int(sample_labels[i])) for i in val_idx]

print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

# ประเมิน mean/std จาก subset ของ train (ไม่โหลดทั้งชุดเพราะกิน RAM) -- สุ่มมาสัก 3000 ภาพพอ
rng = np.random.default_rng(42)
stats_idx = rng.choice(len(train_samples), size=min(3000, len(train_samples)), replace=False)
pixel_sum, pixel_sq_sum, pixel_count = 0.0, 0.0, 0
for i in tqdm(stats_idx, desc="Estimating mean/std"):
    path, _, _ = train_samples[i]
    img = imread(path, as_gray=True)
    img = resize(img, (IMG_SIZE, IMG_SIZE), preserve_range=True)
    if img.max() > 1.0:
        img = img / 255.0
    pixel_sum += img.sum()
    pixel_sq_sum += (img ** 2).sum()
    pixel_count += img.size

DATA_MEAN = float(pixel_sum / pixel_count)
DATA_STD = float(np.sqrt(pixel_sq_sum / pixel_count - DATA_MEAN ** 2))
print(f"mean={DATA_MEAN:.4f}, std={DATA_STD:.4f} (ประเมินจาก {len(stats_idx)} ภาพตัวอย่างของ train)")


class ThaiCharDataset(torch.utils.data.Dataset):
    """โหลด + resize + normalize ทีละภาพตอนถูกเรียก (lazy) แทนการพรีโหลดทั้งหมดเข้า RAM"""

    def __init__(self, samples, img_size, mean, std):
        self.samples = samples
        self.img_size = img_size
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, do_augment, label = self.samples[idx]
        img = imread(path, as_gray=True)
        img = resize(img, (self.img_size, self.img_size), preserve_range=True)
        if img.max() > 1.0:
            img = img / 255.0
        if do_augment:
            img = augment_image(img)  # random ใหม่ทุกครั้งที่ถูกเรียก = augmentation หลากหลายขึ้นทุก epoch
        img = (img.astype('float32') - self.mean) / self.std
        tensor = torch.from_numpy(img.astype('float32')).unsqueeze(0)  # (1, H, W)
        return tensor, label


train_dataset = ThaiCharDataset(train_samples, IMG_SIZE, DATA_MEAN, DATA_STD)
val_dataset = ThaiCharDataset(val_samples, IMG_SIZE, DATA_MEAN, DATA_STD)
print(f"train_dataset: {len(train_dataset)}, val_dataset: {len(val_dataset)}")

Model Loader

In [ ]:
%%writefile Net.py
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class Net(nn.Module):
    def __init__(self, num_classes=72, dropout=0.3):
        super().__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # ไม่ตัด maxpool ออกรอบนี้ (ต่างจาก round 2): input 224x224 ตรงกับที่ ResNet18 pretrained มา
        # downsample เต็ม path (conv1+maxpool+4 stage) ได้ feature map 7x7 ก่อน avgpool ตามมาตรฐาน ไม่ยุบเกินไป
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.fc.in_features, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
from Net import Net
model = Net()
print(model)

Defining Learning Algorithm

In [ ]:
# force using 'cuda'
device = torch.device('cuda')

# defining the model
model = Net(num_classes=len(classes)).to(device)

# defining the optimizer (เพิ่ม weight_decay กัน overfit)
optimizer = Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

# LR scheduler: ลด lr ลงครึ่งนึงเมื่อ val_loss ไม่ลดลง 3 epoch ติดกัน
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# class weight กัน class imbalance (คลาสที่มีตัวอย่างเยอะไม่ dominate loss มากไป)
train_labels_tensor = torch.tensor([s[2] for s in train_samples], dtype=torch.long)
class_counts = torch.bincount(train_labels_tensor, minlength=len(classes)).float()
class_weights = (class_counts.sum() / (len(classes) * class_counts)).to(device)

# defining the loss function
criterion = CrossEntropyLoss(weight=class_weights)

print(model)

Training Model

In [ ]:
# empty list to store training losses/accuracy
train_losses = []
train_accuracies = []

# empty list to store validation losses/accuracy
val_losses = []
val_accuracies = []

# n_epochs เป็นแค่เพดานบน ให้ early stopping ตัดจบเองตาม val_acc จริง
n_epochs = 60
EARLY_STOP_PATIENCE = 10

from torch.utils.data import DataLoader

# num_workers>0 ให้หลาย process ช่วยโหลด/resize ภาพขนาน ๆ กันระหว่างที่ GPU กำลังเทรน batch ก่อนหน้า
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, num_workers=2)

best_val_acc = 0.0
epochs_no_improve = 0
start_epoch = 0

# เซฟ checkpoint เต็ม (model+optimizer+scheduler+ประวัติ) ไป Google Drive ทุก epoch
# กัน session หลุด (Colab ฟรีตัดการเชื่อมต่อเองได้ ไม่เกี่ยวกับ RAM) แล้วต้องเริ่มนับ epoch ใหม่จาก 0
DRIVE_CKPT_PATH = "/content/drive/MyDrive/Deep Learning/checkpoint_round3_224.pt"
DRIVE_BEST_PATH = "/content/drive/MyDrive/Deep Learning/model_best_round3_224.pt"
os.makedirs(os.path.dirname(DRIVE_CKPT_PATH), exist_ok=True)

if os.path.isfile(DRIVE_CKPT_PATH):
    ckpt = torch.load(DRIVE_CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch = ckpt['epoch'] + 1
    best_val_acc = ckpt['best_val_acc']
    epochs_no_improve = ckpt['epochs_no_improve']
    train_losses = ckpt['train_losses']
    train_accuracies = ckpt['train_accuracies']
    val_losses = ckpt['val_losses']
    val_accuracies = ckpt['val_accuracies']
    print(f"พบ checkpoint เดิมที่ epoch {ckpt['epoch']+1} (best_val_acc={best_val_acc*100:.2f}%) "
          f"-> resume ต่อจาก epoch {start_epoch+1}")
else:
    print("ไม่พบ checkpoint เดิมใน Drive เริ่มเทรนใหม่ตั้งแต่ epoch 1")

for epoch in tqdm(range(start_epoch, n_epochs), initial=start_epoch, total=n_epochs):
    model.train()
    tr_loss = 0
    tr_correct = 0
    tr_total = 0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        # clearing the Gradients of the model parameters
        optimizer.zero_grad()

        # prediction for training set
        output_train = model(x_batch)

        loss_train = criterion(output_train, y_batch)
        loss_train.backward()
        optimizer.step()
        tr_loss += loss_train.item()

        # accuracy ของ training batch นี้
        predicted = torch.argmax(output_train, dim=1)
        tr_correct += (predicted == y_batch).sum().item()
        tr_total += y_batch.size(0)

    train_losses.append(tr_loss / len(train_loader))
    train_accuracies.append(tr_correct / tr_total)

    # evaluating performance on the validation set
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            output_val = model(x_batch)
            val_loss += criterion(output_val, y_batch).item()

            # accuracy ของ validation batch นี้
            predicted = torch.argmax(output_val, dim=1)
            val_correct += (predicted == y_batch).sum().item()
            val_total += y_batch.size(0)

    val_losses.append(val_loss / len(val_loader))
    val_accuracies.append(val_correct / val_total)

    # ลด lr เมื่อ val_loss หยุดลดลง
    scheduler.step(val_losses[-1])

    cur_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{n_epochs} - train_loss: {train_losses[-1]:.4f} - train_acc: {train_accuracies[-1]*100:.2f}% - val_loss: {val_losses[-1]:.4f} - val_acc: {val_accuracies[-1]*100:.2f}% - lr: {cur_lr:.6f}")

    # save เฉพาะตอน val_acc ดีขึ้น (กันเก็บโมเดลที่ overfit ตอนท้าย ๆ)
    if val_accuracies[-1] > best_val_acc:
        best_val_acc = val_accuracies[-1]
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'model.pt')
        # เซฟตัวที่ดีที่สุดไป Drive ด้วย (ไม่ใช่แค่ local /content ที่หายตอน session หลุด)
        torch.save(model.state_dict(), DRIVE_BEST_PATH)
    else:
        epochs_no_improve += 1

    # เซฟ checkpoint ไป Drive ทุก epoch (ไม่ใช่แค่ตอน val_acc ดีขึ้น) เพื่อให้ resume ต่อได้แม่นถ้าหลุดกลางคัน
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_val_acc': best_val_acc,
        'epochs_no_improve': epochs_no_improve,
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
    }, DRIVE_CKPT_PATH)

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"Early stopping ที่ epoch {epoch+1} (val_acc ไม่ดีขึ้น {EARLY_STOP_PATIENCE} epoch ติดกัน) best_val_acc={best_val_acc*100:.2f}%")
        break

Save Model

In [ ]:
# model.pt ถูก save ไว้ระหว่างเทรนแล้วทุกครั้งที่ val_acc ดีขึ้น (checkpoint ที่ val_acc สูงสุด)
# ไม่ต้อง save ซ้ำตรงนี้ — เช็คผลลัพธ์สุดท้ายพอ
print(f"เทรนเสร็จ — best_val_acc: {best_val_acc*100:.2f}% (บันทึกไว้ใน model.pt แล้ว)")